In [2]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import GridSearchCV, StratifiedKFold

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer

import pickle

In [ ]:
df = pd.read_csv("Data/train.csv")

In [8]:
y = df.Survived
X = df.drop(columns=['Survived'])

### Cabin feature

In [36]:
df['HasCabin'] = df.Cabin.notna().astype(int)

In [43]:
df['Deck'] = df['Cabin'].str[0].fillna('U')

In [53]:
df['CabinCount'] = (df['Cabin'].str.count(' ')+1).fillna(0).astype('int')

In [55]:
df = df.drop(columns=['Cabin'])

In [56]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked,HasCabin,Deck,CabinCount
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,S,0,U,0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C,1,C,1
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,S,0,U,0
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,S,1,C,1
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,S,0,U,0


In [57]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 14 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Embarked     889 non-null    object 
 11  HasCabin     891 non-null    int64  
 12  Deck         891 non-null    object 
 13  CabinCount   891 non-null    int64  
dtypes: float64(2), int64(7), object(5)
memory usage: 97.6+ KB


### Ticket feature

In [68]:
ticket_counts = df.Ticket.value_counts()
df['GroupSize'] = df.Ticket.map(ticket_counts)

In [ ]:
df['TicketPrefix'] = (df['Ticket']
                        .str.split()
                        .apply(lambda x: x[0].replace('.', '').replace('/', '') 
                        if len(x)>1 else 'NoPrefix'))

In [94]:
df = df.drop(columns=['Ticket'])

In [106]:
df['TicketPrefix'].value_counts()

TicketPrefix
NoPrefix    665
PC           60
CA           41
A5           21
SOTONOQ      15
STONO        12
WC           10
A4            7
SCPARIS       7
STONO2        6
SOC           6
C             5
FCC           5
SCParis       4
SOPP          3
SCAH          3
WEP           3
PP            3
PPP           2
SOTONO2       2
SWPP          2
SP            1
SCA4          1
SCOW          1
SOP           1
Fa            1
AS            1
SC            1
FC            1
CASOTON       1
Name: count, dtype: int64

In [95]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Fare,Embarked,HasCabin,Deck,CabinCount,GroupSize,TicketPrefix
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,7.2500,S,0,U,0,1,A5
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,71.2833,C,1,C,1,1,PC
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,7.9250,S,0,U,0,1,STONO2
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,53.1000,S,1,C,1,2,NoPrefix
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,8.0500,S,0,U,0,1,NoPrefix


In [97]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   891 non-null    int64  
 1   Survived      891 non-null    int64  
 2   Pclass        891 non-null    int64  
 3   Name          891 non-null    object 
 4   Sex           891 non-null    object 
 5   Age           714 non-null    float64
 6   SibSp         891 non-null    int64  
 7   Parch         891 non-null    int64  
 8   Fare          891 non-null    float64
 9   Embarked      889 non-null    object 
 10  HasCabin      891 non-null    int64  
 11  Deck          891 non-null    object 
 12  CabinCount    891 non-null    int64  
 13  GroupSize     891 non-null    int64  
 14  TicketPrefix  891 non-null    object 
dtypes: float64(2), int64(8), object(5)
memory usage: 104.5+ KB


### Name feature

In [100]:
df['NameLen'] = df['Name'].apply(len)